# Day 7 — Pandas II: Data Cleaning
**Dataset:** `Day13_Sales_100_Records.csv` (keep in same folder as notebook)  
**Deliverable:** A cleaned dataset and a notebook documenting cleaning decisions.

In [1]:
import pandas as pd

df = pd.read_csv("Day13_Sales_100_Records.csv")
print("Before cleaning:", df.shape)
display(df.head())

Before cleaning: (100, 11)


,OrderID,OrderDate,Product,Category,Region,SalesChannel,CustomerSegment,Quantity,UnitPrice,DiscountPct,Sales
0,ORD001,2026-09-24,Headphones,Accessories,Mumbai,Online,Corporate,3,2700,0.00,8100.0
1,ORD002,2026-10-17,Keyboard,Accessories,Mumbai,Phone,Small Business,2,1100,0.15,1870.0
2,ORD003,2026-07-02,Laptop,Electronics,Nashik,Online,Individual,1,53000,0.15,45050.0
3,ORD004,2026-10-10,Headphones,Accessories,Nashik,Retail Store,Individual,7,3700,0.00,25900.0
4,ORD005,2026-09-24,Printer,Electronics,Nashik,Retail Store,Corporate,3,13500,0.10,36450.0


## 1. Profile data quality before cleaning

In [2]:
before_shape = df.shape
print("Missing values:\n", df.isna().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Duplicate OrderIDs:", df["OrderID"].duplicated().sum())
print("Data types:\n", df.dtypes)

Missing values:
 OrderID            0
OrderDate          0
Product            0
Category           0
Region             0
SalesChannel       0
CustomerSegment    0
Quantity           0
UnitPrice          0
DiscountPct        0
Sales              0
dtype: int64
Exact duplicates: 0
Duplicate OrderIDs: 0
Data types:
 OrderID                str
OrderDate              str
Product                str
Category               str
Region                 str
SalesChannel           str
CustomerSegment        str
Quantity             int64
UnitPrice            int64
DiscountPct        float64
Sales              float64
dtype: object


## 2. Apply cleaning decisions

In [3]:
clean = df.copy()

# Standardize text fields by removing extra spaces.
text_cols = ["OrderID", "Product", "Category", "Region", "SalesChannel", "CustomerSegment"]
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

# Convert date and numeric columns safely.
clean["OrderDate"] = pd.to_datetime(clean["OrderDate"], errors="coerce")
numeric_cols = ["Quantity", "UnitPrice", "DiscountPct", "Sales"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Remove exact duplicate rows only. Do not automatically drop rows just because OrderID repeats;
# investigate those separately in a real dataset.
clean = clean.drop_duplicates()

print("After cleaning:", clean.shape)
display(clean.head())

After cleaning: (100, 11)


,OrderID,OrderDate,Product,Category,Region,SalesChannel,CustomerSegment,Quantity,UnitPrice,DiscountPct,Sales
0,ORD001,2026-09-24,Headphones,Accessories,Mumbai,Online,Corporate,3,2700,0.00,8100.0
1,ORD002,2026-10-17,Keyboard,Accessories,Mumbai,Phone,Small Business,2,1100,0.15,1870.0
2,ORD003,2026-07-02,Laptop,Electronics,Nashik,Online,Individual,1,53000,0.15,45050.0
3,ORD004,2026-10-10,Headphones,Accessories,Nashik,Retail Store,Individual,7,3700,0.00,25900.0
4,ORD005,2026-09-24,Printer,Electronics,Nashik,Retail Store,Corporate,3,13500,0.10,36450.0


## 3. Recheck the cleaned dataset

In [4]:
print("Missing values after cleaning:\n", clean.isna().sum())
print("Exact duplicate rows after cleaning:", clean.duplicated().sum())
print("Date type:", clean["OrderDate"].dtype)
print("Numeric types:\n", clean[numeric_cols].dtypes)
print("Non-positive quantities:", (clean["Quantity"] <= 0).sum())

Missing values after cleaning:
 OrderID            0
OrderDate          0
Product            0
Category           0
Region             0
SalesChannel       0
CustomerSegment    0
Quantity           0
UnitPrice          0
DiscountPct        0
Sales              0
dtype: int64
Exact duplicate rows after cleaning: 0
Date type: datetime64[us]
Numeric types:
 Quantity         int64
UnitPrice        int64
DiscountPct    float64
Sales          float64
dtype: object
Non-positive quantities: 0


## 4. Save cleaned data

In [5]:
clean.to_csv("Day07_Cleaned_Sales.csv", index=False)
print("Saved Day07_Cleaned_Sales.csv")

Saved Day07_Cleaned_Sales.csv


## 5. Cleaning decisions and limitations
- Converted OrderDate to datetime; invalid dates become missing values and should be investigated.
- Converted numeric fields using `errors="coerce"`; values that cannot be parsed become missing.
- Trimmed extra spaces in text fields.
- Removed exact duplicate rows.
- Did not automatically remove duplicate OrderIDs because repeated IDs require business-rule investigation.
- Did not impute missing values because the sample data and business rules do not justify a specific imputation.

Write a short conclusion comparing the data before and after cleaning.